In [1]:
import os
from pathlib import Path
import json
from jsonargparse import CLI
import boto3

import time
from copy import deepcopy
import threading
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextGenerationPipeline
from peft import PeftModel

def get_batch_response(text, base_model, model_path, temperature, max_tokens, EXSTING):
    """
    Modified get_response function to use local model instead of API calls.
    model_path: path to the local model directory
    tokenizer and model: optional pre-loaded tokenizer and model objects
    """
    
    while True:
        try:
            # content = prompt.format(text)
            
            # Load model and tokenizer if not provided (for thread safety)
            
            # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            
            # Load tokenizer
            tokenizer = AutoTokenizer.from_pretrained(base_model)
            base_model = AutoModelForCausalLM.from_pretrained(base_model,  
                                torch_dtype=torch.float16,
                                device_map='auto'
                                )

            # Load LoRA adapter on top of base model
            model = PeftModel.from_pretrained(base_model, model_path)

            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            model.eval()
            results=[]

            pipe = TextGenerationPipeline(model=model, tokenizer=tokenizer)

            results = pipe(text, max_new_tokens=max_tokens,
                            temperature=temperature,
                            do_sample=temperature > 0,
                            pad_token_id=tokenizer.pad_token_id,
                            eos_token_id=tokenizer.eos_token_id,
                            use_cache=True,
                            batch_size=len(text)
                        )

            results = [result['generated_text'][len(prompt):].strip() if result['generated_text'].startswith(prompt) else result['generated_text']
                                for prompt, result in zip(text, results)]
            

            # for content in text:
            #     # Tokenize input
            #     inputs = tokenizer(
            #         content,
            #         return_tensors="pt",
            #         truncation=True,
            #         max_length=512,
            #         padding=True
            #     )
            
            #     # Move to device
            #     device = next(model.parameters()).device
            #     inputs = {k: v.to(device) for k, v in inputs.items()}
            
            #     # Generate response
            #     with torch.no_grad():
            #         outputs = model.generate(
            #             **inputs,
            #             max_new_tokens=max_tokens,
            #             temperature=temperature,
            #             do_sample=temperature > 0,
            #             pad_token_id=tokenizer.pad_token_id,
            #             eos_token_id=tokenizer.eos_token_id,
            #             use_cache=True
            #         )
            
            #     # Decode response (only the new tokens)
            #     input_length = inputs['input_ids'].shape[1]
            #     generated_tokens = outputs[0][input_length:]
            #     response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
            #     results.append(response_text)
            
            
            return results
            
            
        except Exception as e:
            if e == KeyboardInterrupt:
                raise e
            print(f"Error: {e}")
            time.sleep(2)
            continue

        break


def main(from_json: str = None, to_json: str = None, prompt: str = None, base_model: str = 'llama-3.1-instruct',
         model_path: str = 'llama-3.1-instruct', temperature: float = 0, max_tokens: int = 512, 
         batch_size: int = 8, n_print: int = 100, n_samples: int = -1, 
         input_field: str = 'input', existing_json: str = None):
    EXSTING = {}
    if existing_json is not None:
        with open(existing_json, 'r') as f:
            for l in f.readlines():
                d = json.loads(l)
                if d['resp'] != 'API Failed':
                    EXSTING[d['prompt']] = d
    
    
    path = Path(to_json)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.touch()

    with open(from_json, "r") as fr, open(to_json, 'w') as fw:

        results = []

        lines = fr.readlines()
        total_lines = min(len(lines), n_samples) if n_samples > 0 else len(lines)
        start_time = time.time()
        
        for i in range(0, len(lines), batch_size):
           
            batch = [prompt.format(json.loads(lines[i+j])[input_field]) for j in range(min(batch_size, len(lines)-i))]
            
            batch_results = get_batch_response(
                                batch, base_model, model_path, temperature, max_tokens, EXSTING
                            )
            
            for result in batch_results:
                fw.write(json.dumps(result) + '\n')

            if i % n_print == 0:
                print(f'Time elapsed: {time.time() - start_time:.2f} sec. {i+8} / {total_lines} samples generated. ')

main(from_json='testsets/inspired/test_clean.jsonl',
    to_json='test_res/inspired/llama-3.2-instruct/inspired_test_clean.jsonl',
    prompt="Pretend you are a movie recommender system. I will give you a conversation between a user and you (a recommender system). Here is the conversation: {} Based on the conversation, you reply with a list of 20 recommendations in the format of '1. [Movie Name]\n 2. [Movie Name]\n ...' without extra sentences. ",
    base_model='meta-llama/Llama-3.2-1B-Instruct',
    model_path='../outputs/sft/inspired/test',
    temperature=0.1,
    max_tokens=512,
    n_print=1,
    n_samples=-1)

/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cuda:7
The model 'PeftModelForCausalLM' is not supported for . Supported models are ['AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausa

Error: list indices must be integers or slices, not str
Error: Repo id must use alphanumeric chars or '-', '_', '.', '--' and '..' are forbidden, '-' and '.' cannot start or end the name, max length is 96: 'LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=32, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=32, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
  

KeyboardInterrupt: 